# Reinforcement Learning Approach

In [10]:
# load data
import pandas as pd

df = pd.read_csv("wide_processed_final.csv")
df.head()

,word1,word2,word3,word4,category,difficulty,General Category
0,curses,fudge,blast,crud,"""aw, heck!""",medium,Idioms & Slang
1,choral,jazz,rap,americana,"""best ___ performance"" grammy award",very_hard,Fill-in-the-Blank
2,lord,please,sheesh,brother,"""give me a break!""",easy,Idioms & Slang
3,heavens,gracious,mercy,dear,"""my goodness!""",easy,Idioms & Slang
4,piece of cake,no sweat,easy,child’s play,"""nothing to it!""",easy,Idioms & Slang


In [22]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import pickle

print("Loading dataset...")
df = pd.read_csv("wide_processed_final.csv")

# Extract vocabulary
words = pd.concat([df['word1'], df['word2'], df['word3'], df['word4']])
words = words.astype(str).str.strip().str.lower().unique().tolist()
print(f"Found {len(words)} unique words.")

# Load the fast transformer
print("Loading SentenceTransformer ('all-MiniLM-L6-v2')...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# Encode words
print("Encoding... (Takes about 1-2 minutes)")
embeddings = model.encode(words, show_progress_bar=True, convert_to_numpy=True)

word_to_vec = {word: embeddings[i] for i, word in enumerate(words)}

# Save to pickle
with open("word_embeddings_dict.pkl", 'wb') as f:
    pickle.dump(word_to_vec, f)
print("Saved word_embeddings_dict.pkl successfully!")

KeyboardInterrupt: 

In [31]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import pandas as pd
import random
import pickle
import itertools

import torch
import torch.nn as nn
import torch.nn.functional as F

from stable_baselines3 import PPO
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.callbacks import BaseCallback
from typing import Callable
from IPython.display import clear_output

In [36]:
class SmartConnectionsFeatureExtractor(BaseFeaturesExtractor):
    def __init__(self, observation_space, features_dim=256):
        super(SmartConnectionsFeatureExtractor, self).__init__(observation_space, features_dim)
        
        # 256 (16x16 Matrix) + 1 (Lives) + 16 (Current Selection Array) = 273 inputs
        input_size = 277 
        
        self.linear = nn.Sequential(
            nn.Linear(input_size, 512),
            nn.ReLU(),
            nn.Linear(512, features_dim),
            nn.ReLU()
        )

    def forward(self, observations):
        board_vectors = observations["board_vectors"]  
        batch_size = board_vectors.shape[0]
        
        # 1. Normalize the vectors
        norm_board = F.normalize(board_vectors, p=2, dim=2)
        
        # 2. Multiply to get the 16x16 similarity grid
        sim_matrix = torch.bmm(norm_board, norm_board.transpose(1, 2))
        
        # 3. Mask out the diagonal (so it can't see a word's similarity to itself)
        mask = torch.eye(16, device=sim_matrix.device).bool()
        mask = mask.unsqueeze(0).expand(batch_size, -1, -1)
        sim_matrix.masked_fill_(mask, 0.0)
        
        # 4. Flatten the matrix and extract the other state variables
        flat_sim = sim_matrix.view(batch_size, -1)
        lives = observations["lives_remaining"].float().view(batch_size, -1)
        selection = observations["current_selection"].float().view(batch_size, -1)
        
        # 5. Glue the Matrix, Lives, and Selection together into one array
        rl_state = torch.cat([flat_sim, lives, selection], dim=1)
        
        return self.linear(rl_state)

In [56]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import pandas as pd
import random
import pickle

class ConnectionsEnv(gym.Env):
    def __init__(self, csv_path="wide_processed_final.csv", dict_path="word_embeddings_dict.pkl"):
        super(ConnectionsEnv, self).__init__()
        
        self.df = pd.read_csv(csv_path)
        with open(dict_path, 'rb') as f:
            self.word_to_vec = pickle.load(f)
            
        # The Human Controller: 16 simple buttons
        self.action_space = spaces.Discrete(16)
        
        # The Dynamic Focus Architecture
        self.observation_space = spaces.Dict({
            "active_similarities": spaces.Box(low=-1.0, high=1.0, shape=(16,), dtype=np.float32),
            "valid_mask": spaces.MultiBinary(16), 
            "lives_remaining": spaces.Discrete(5)
        })
        
        self.current_board = []
        self.solution_groups = []
        self.remaining_indices = []
        self.lives = 4
        self.previous_guesses = set()
        self.current_selection = [] 
        
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        
        while True:
            sampled_rows = self.df.sample(4)
            self.solution_groups = []
            words_pool = []
            valid_board = True
            
            for _, row in sampled_rows.iterrows():
                group = {str(row['word1']).strip().lower(), 
                         str(row['word2']).strip().lower(), 
                         str(row['word3']).strip().lower(), 
                         str(row['word4']).strip().lower()}
                
                if len(group) != 4:
                    valid_board = False
                    break
                    
                self.solution_groups.append(group)
                words_pool.extend(list(group))
                
            if valid_board and len(set(words_pool)) == 16:
                break
                
        random.shuffle(words_pool)
        self.current_board = words_pool
        self.remaining_indices = list(range(16))
        self.lives = 4
        self.previous_guesses = set()
        self.current_selection = []
        
        return self._get_obs(), {}
        
    def _get_obs(self):
        vectors = np.zeros((16, 384), dtype=np.float32)
        valid_mask = np.zeros(16, dtype=np.int8)
        active_similarities = np.zeros(16, dtype=np.float32)
        
        # 1. Build the vectors and the valid mask
        for i, word in enumerate(self.current_board):
            vectors[i] = self.word_to_vec.get(word, np.zeros(384))
            if i in self.remaining_indices and i not in self.current_selection:
                valid_mask[i] = 1

        # 2. Dynamic Focus: Calculate similarity to whatever is currently clicked
        if len(self.current_selection) > 0:
            selected_vectors = [vectors[i] for i in self.current_selection]
            mean_vector = np.mean(selected_vectors, axis=0)
            norm_mean = np.linalg.norm(mean_vector)
            
            if norm_mean > 0:
                for i in range(16):
                    norm_v = np.linalg.norm(vectors[i])
                    if norm_v > 0:
                        sim = np.dot(mean_vector, vectors[i]) / (norm_mean * norm_v)
                        active_similarities[i] = float(sim)
                        
        return {
            "active_similarities": active_similarities,
            "valid_mask": valid_mask,
            "lives_remaining": self.lives
        }

    def step(self, action):
        # 1. Did it click a removed word?
        if action not in self.remaining_indices:
            return self._get_obs(), -2, False, False, {"info": "Clicked removed word"}
            
        # 2. Did it click a word it ALREADY selected this turn?
        if action in self.current_selection:
            return self._get_obs(), -2, False, False, {"info": "Clicked already selected word"}
            
        # 3. Valid click! Add it to the buffer.
        self.current_selection.append(action)
        
        # --- THE FIX: IMMEDIATE MICRO-REWARDS ---
        step_reward = 0
        if len(self.current_selection) > 1:
            first_word = self.current_board[self.current_selection[0]]
            newest_word = self.current_board[action]
            
            for group in self.solution_groups:
                if first_word in group:
                    if newest_word in group:
                        step_reward = 0.5  # Good click!
                    else:
                        step_reward = -0.5 # Bad click!
                    break
                    
        # 4. Is the turn over yet? (Waiting for 4 words)
        if len(self.current_selection) < 4:
            return self._get_obs(), step_reward, False, False, {"info": "Valid click"}
            
        # 5. WE HAVE 4 WORDS! Evaluate the final group.
        guess_indices = frozenset(self.current_selection)
        self.current_selection = []  # Clear the buffer
        
        # --- THE FIX: IMMORTAL GROUNDHOG ---
        if guess_indices in self.previous_guesses:
            self.lives -= 1 # Actually deduct the life!
            if self.lives <= 0:
                return self._get_obs(), -15, True, False, {"info": "Game Over: Repeated guess"}
            return self._get_obs(), -10, False, False, {"info": "Repeated guess"}
            
        self.previous_guesses.add(guess_indices)
        guessed_words = {self.current_board[i] for i in guess_indices}
        
        # Start the final reward with whatever the 4th click earned
        reward = step_reward 
        terminated = False
        info = {}
        max_overlap = 0
        matched_group = None
        
        for group in self.solution_groups:
            overlap = len(guessed_words.intersection(group))
            if overlap > max_overlap:
                max_overlap = overlap
                matched_group = group

        if max_overlap == 4:
            reward += 20
            self.remaining_indices = [i for i in self.remaining_indices if i not in guess_indices]
            self.solution_groups.remove(matched_group)
            info["status"] = "Correct!"
            
            if len(self.remaining_indices) == 0:
                reward += 30
                terminated = True
                info["status"] = "Game Won!"
                
        elif max_overlap == 3:
            reward += 3
            self.lives -= 1
            info["status"] = "One Away!"
            
        elif max_overlap == 2:
            reward -= 1 
            self.lives -= 1
            info["status"] = "Two Away!"
            
        else:
            reward -= 1
            self.lives -= 1
            info["status"] = "Incorrect."
            
        if self.lives <= 0 and not terminated:
            reward -= 15
            terminated = True
            info["status"] = "Game Over"
            
        return self._get_obs(), reward, terminated, False, info

In [38]:
from stable_baselines3.common.callbacks import BaseCallback
from IPython.display import clear_output

class CleanOutputCallback(BaseCallback):
    """
    Custom callback that intercepts the training loop and clears 
    the Jupyter output right before printing the newest log table.
    """
    def _on_rollout_end(self) -> bool:
        # wait=True prevents screen flickering by waiting for the 
        # new text to be ready before deleting the old text
        clear_output(wait=True)
        return True

    def _on_step(self) -> bool:
        # We don't need to do anything on individual steps
        return True

In [52]:
from stable_baselines3 import PPO

print("Initializing Dynamic Focus Environment...")
env = ConnectionsEnv()

print("Initializing Default PPO Agent...")
agent = PPO(
    "MultiInputPolicy", # This automatically flattens our 33 inputs perfectly!
    env, 
    learning_rate=0.0003, 
    ent_coef=0.01,         
    target_kl=0.05,        
    n_steps=2048, # Sped up batching since the math is simpler         
    batch_size=256,                        
    verbose=1,
    tensorboard_log="./connections_tensorboard/"
)

print("Starting Master Run...")
agent.learn(total_timesteps=1_000_000, tb_log_name="Dynamic_Focus_Run", callback=CleanOutputCallback())

print("Saving Agent...")
agent.save("nyt_connections_dynamic_final")

-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 16          |
|    ep_rew_mean          | -16.7       |
| time/                   |             |
|    fps                  | 1615        |
|    iterations           | 489         |
|    time_elapsed         | 619         |
|    total_timesteps      | 1001472     |
| train/                  |             |
|    approx_kl            | 0.003473274 |
|    clip_fraction        | 0.0333      |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.641      |
|    explained_variance   | 0.0153      |
|    learning_rate        | 0.0003      |
|    loss                 | 1.1         |
|    n_updates            | 4880        |
|    policy_gradient_loss | -0.00167    |
|    value_loss           | 2.24        |
-----------------------------------------
Saving Agent...


In [48]:
from stable_baselines3 import PPO

print("Loading stuck agent...")
# Load the agent that is currently trapped at -17 or -5
agent = PPO.load("nyt_connections", env=env)

# --- INITIATE EXPLORATION SHOCK ---
agent.ent_coef = 0.1        # MASSIVE increase in random exploration (was 0.01)
agent.learning_rate = 0.0005 # Faster learning to rewrite bad habits
agent.target_kl = 0.15      # Allow the network to take bigger leaps in logic

print("Starting Exploration Phase (250,000 steps)...")
# Let it wander and try crazy combinations
agent.learn(total_timesteps=500_000, tb_log_name="Exploration_Shock", reset_num_timesteps=False, callback=CleanOutputCallback())

# --- THE COOL DOWN ---
print("Cooling down and locking in new knowledge...")
agent.ent_coef = 0.01       # Return to focused execution
agent.learning_rate = 0.0001
agent.target_kl = 0.05      

# Let it fine-tune the new strategies it just found
agent.learn(total_timesteps=1_000_000, tb_log_name="Post_Shock_Recovery", reset_num_timesteps=False, callback=CleanOutputCallback())

print("Saving Rescued Agent...")
agent.save("nyt_connections_rescued_agent")

------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 16.2         |
|    ep_rew_mean          | -19.6        |
| time/                   |              |
|    fps                  | 1174         |
|    iterations           | 89           |
|    time_elapsed         | 310          |
|    total_timesteps      | 2871296      |
| train/                  |              |
|    approx_kl            | 0.0147569105 |
|    clip_fraction        | 0.183        |
|    clip_range           | 0.2          |
|    entropy_loss         | -2.31        |
|    explained_variance   | -0.0308      |
|    learning_rate        | 5.47e-05     |
|    loss                 | 2.14         |
|    n_updates            | 7000         |
|    policy_gradient_loss | -0.013       |
|    value_loss           | 5.35         |
------------------------------------------


KeyboardInterrupt: 

In [ ]:
# run "tensorboard --logdir ./connections_tensorboard/ --port 6006" for graphs

# looking at ep_rew_mean to increase to ideally 60, this is the "score"

# entropy loss should ideally converging near zero, this is the randomness of the model

# ep_len_mean should be nearing 5-6, this goes in hand with score, shows how many guesses the model took

# approx_kl should be a stable bounce near 0.01, big jumps to 0.1 might indicate LR is too high

# training documentation

at LR = 0.00005, the approx_kl starts off rocky at high values near 0.1 but gradually fall off

entropy_loss rises to values -11, but gradually plateaus at -10.5 which is not ideal

ep_len_mean spikes from 5.9 to 6.2 which are the values we ideally want

ep_rew_mean is shaky at -22 which indicate a failure in testing

Step 120k seems to be where the model starts to converge quickly with all values spiking towards the optimal value, with the exception of ep_rew_mean. A tweak in the reward scaling might be needed.

values seem to fall after steps 160k, with a return to initial values before the spike.

In [1]:
import numpy as np

# 1. Start a fresh game in your environment
env = ConnectionsEnv()
obs, info = env.reset()
done = False
total_score = 0
turn_count = 1

print("="*40)
print("HEURISTIC BOT PLAYING")
print("="*40)

while not done:
    # 2. If we haven't clicked anything yet, just pick the first valid word to start a group
    if len(env.current_selection) == 0:
        valid_indices = [i for i in range(16) if obs["valid_mask"][i] == 1]
        action = valid_indices[0] 
        print(f"\n--- Group {turn_count} ---")
        turn_count += 1
        
    # 3. If we HAVE a word selected, find the highest similarity score!
    else:
        sims = obs["active_similarities"]
        mask = obs["valid_mask"]
        
        # Hide the invalid buttons by giving them a terrible score (-999)
        masked_sims = np.where(mask == 1, sims, -999)
        
        # Click the button with the highest remaining score (argmax!)
        action = int(np.argmax(masked_sims))
        
    # 4. Take the step
    word_clicked = env.current_board[action]
    print(f"Bot clicked: {word_clicked}")
    
    obs, reward, terminated, truncated, info = env.step(action)
    total_score += reward
    
    if info.get("status") in ["Correct!", "Game Won!", "One Away!", "Two Away!", "Incorrect."]:
        print(f">> Result: {info['status']} | Reward: {reward}")

    if terminated:
        done = True

print("="*40)
print(f"FINAL SCORE: {total_score}")

NameError: name 'ConnectionsEnv' is not defined